# 第 7 章 · 多 agent（agent-as-tool）

把**一个 agent 包装成工具**，给另一个 agent 调用。

和 Coordinator 的区别：那里是"把对话整个转交"，这里是"像调函数一样调，拿回值继续干自己的活"。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

try:
    from google.adk.runners import InMemoryRunner
    from google.genai import types
except ModuleNotFoundError:
    # 【适配】内核选错时的提示；比裸 ModuleNotFoundError 好定位
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool


def generate_image(prompt: str) -> dict:
    """Generates an image based on a textual prompt.

    Args:
        prompt: A detailed description of the image to generate.

    Returns:
        A dictionary with the status and a short description of the image.
    """
    # 【适配】书里返回 mock 的 image bytes；这里返回文字描述，方便在 notebook 里看结果
    print(f"TOOL: Generating image for prompt: '{prompt}'")
    return {"status": "success", "description": f"已生成图片：{prompt}"}


image_generator_agent = LlmAgent(
    name="ImageGen",
    model=adk_model(),                      # 【适配】gemini-2.0-flash 已下线
    description="Generates an image based on a detailed text prompt.",
    instruction="你是图像生成专家。把用户的整个请求当作 prompt 传给 generate_image 工具，"
                "然后把工具返回的描述原样告诉我。",
    tools=[generate_image],
)

# ★ 本章新增 AgentTool：把一个 agent 变成另一个 agent 可调用的工具
# 【适配】ADK 2.13 的 AgentTool 不收 description=，描述改由子 agent 自己的 description 提供
image_tool = agent_tool.AgentTool(agent=image_generator_agent)

artist_agent = LlmAgent(
    name="Artist",
    model=adk_model(),                      # 【适配】同上
    instruction="你是创意艺术家。先想一个有创意的画面描述，然后用 ImageGen 工具生成它。",
    tools=[image_tool],
)

finals, state = run_once(artist_agent, "画一只戴礼帽的猫")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:400]}")

TOOL: Generating image for prompt: 'A high-quality, close-up photograph of a handsome British Shorthair cat. The cat is grey with plush, thick fur and striking green eyes, looking directly into the camera with a poised and dignified expression. On its head, it wears a miniature, perfectly proportioned black silk top hat, complete with a thin, elegant crimson ribbon. The setting is a cozy, warmly lit library with shelves of old leather-bound books softly blurred in the background. The lighting is warm and dramatic, highlighting the texture of the cat's fur and the subtle sheen of the top hat, creating a sophisticated and whimsical portrait.'



--- [Artist] ---
这是一幅为您精心创作的“戴礼帽的猫”的画面。

我将它设定为一只高贵的英国短毛猫，身处一间温馨、充满书香气的图书馆中。它戴着一顶精致的黑色丝绸小礼帽，上面还装饰着一根深红色的丝带。猫咪用它那双明亮的绿色眼睛自信地看着镜头，显得既优雅又有些俏皮。希望您会喜欢这个充满绅士风范的猫咪形象。
